# Analyse du Data Drift — Modèle de scoring crédit

Ce notebook compare deux fenêtres temporelles de données de production :
- **Référence** : trafic simulé sans drift (distributions proches de l'entraînement)
- **Courant** : trafic simulé avec drift (distributions décalées pour simuler une dérive)

Les données sont chargées depuis la base PostgreSQL loggée par l'API.

## 1. Imports et configuration

In [1]:
import warnings
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display
from evidently import Dataset, Report
from evidently.presets import DataDriftPreset

import sys
sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from app.database import get_engine

warnings.filterwarnings("ignore")

# Features suivies (celles décalées par le script de simulation avec --drift)
FEATURES = [
    "DAYS_BIRTH",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "score",
]

REPORT_PATH = Path("reports/drift_report.html")
REPORT_PATH.parent.mkdir(exist_ok=True)

print("Configuration OK")

Configuration OK


## 2. Chargement des données depuis PostgreSQL

In [2]:
engine = get_engine()

cols = ", ".join(f'"{f}"' for f in FEATURES)

# Chargement de toutes les données avec timestamps pour détecter le gap
all_data = pd.read_sql(
    f'SELECT {cols}, "timestamp" FROM predictions ORDER BY "timestamp"',
    engine,
)
all_data["timestamp"] = pd.to_datetime(all_data["timestamp"], utc=True)
all_data["delta"] = all_data["timestamp"].diff()

# Le plus grand saut temporel = frontière entre les deux batches de simulation
cutoff_row = all_data["delta"].idxmax()
CUTOFF = all_data.loc[cutoff_row, "timestamp"]
gap_max = all_data["delta"].max()

assert gap_max > pd.Timedelta("1min"), (
    f"Pas de gap significatif détecté ({gap_max}). "
    "Relance les deux batches de simulation avec un délai entre eux."
)

reference = all_data[all_data["timestamp"] < CUTOFF][FEATURES].reset_index(drop=True)
current   = all_data[all_data["timestamp"] >= CUTOFF][FEATURES].reset_index(drop=True)

print(f"Coupure détectée automatiquement : {CUTOFF}")
print(f"Gap maximal entre deux requêtes  : {gap_max}")
print(f"Référence : {len(reference)} lignes")
print(f"Courant   : {len(current)} lignes")
display(reference.describe().round(2))

Coupure détectée automatiquement : 2026-04-27 11:02:04.276720+00:00
Gap maximal entre deux requêtes  : 0 days 03:39:30.062322
Référence : 101 lignes
Courant   : 100 lignes


,DAYS_BIRTH,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,score
count,101.00,101.00,101.00,101.00,46.00,101.00,83.00,101.00
mean,-15965.74,164095.58,634382.75,25273.82,0.49,0.50,0.51,0.33
std,4962.02,87650.10,507750.05,21646.37,0.22,0.19,0.22,0.17
min,-24767.80,1.00,1.00,1.00,0.00,0.00,0.00,0.06
25%,-20578.90,102278.84,294745.36,10094.20,0.33,0.39,0.38,0.20
50%,-15973.00,148340.26,498833.58,19322.56,0.49,0.50,0.49,0.28
75%,-11535.80,202826.63,895001.46,34776.74,0.63,0.65,0.67,0.43
max,-1.00,632181.13,3596562.08,151601.72,0.93,0.99,0.99,0.88


## 3. Rapport Evidently — Détection de drift

In [3]:
report = Report([DataDriftPreset()])
result = report.run(
    Dataset.from_pandas(reference),
    Dataset.from_pandas(current),
)

result.save_html(str(REPORT_PATH))
print(f"Rapport exporté : {REPORT_PATH.resolve()}")

Rapport exporté : /home/rapha/ai-engineer/credit-scoring-mlops-production/monitoring/reports/drift_report.html


In [4]:
summary = pd.DataFrame({
    "mean_ref":    reference.mean(numeric_only=True).round(3),
    "mean_cur":    current.mean(numeric_only=True).round(3),
    "std_ref":     reference.std(numeric_only=True).round(3),
    "std_cur":     current.std(numeric_only=True).round(3),
})
summary["delta_mean_%"] = ((summary["mean_cur"] - summary["mean_ref"]) / summary["mean_ref"].abs() * 100).round(1)

display(summary)
print(f"\nRapport interactif complet : ouvrir dans un navigateur")
print(REPORT_PATH.resolve())

,mean_ref,mean_cur,std_ref,std_cur,delta_mean_%
DAYS_BIRTH,-15965.743,-20397.954,4962.022,4215.892,-27.8
AMT_INCOME_TOTAL,164095.579,116746.778,87650.099,52673.533,-28.9
AMT_CREDIT,634382.753,624812.338,507750.046,345895.216,-1.5
AMT_ANNUITY,25273.818,41221.083,21646.370,24384.600,63.1
EXT_SOURCE_1,0.494,0.384,0.219,0.202,-22.3
EXT_SOURCE_2,0.504,0.323,0.195,0.164,-35.9
EXT_SOURCE_3,0.513,0.350,0.220,0.156,-31.8
score,0.331,0.508,0.167,0.173,53.5



Rapport interactif complet : ouvrir dans un navigateur
/home/rapha/ai-engineer/credit-scoring-mlops-production/monitoring/reports/drift_report.html


## 4. Interprétation

### Features en drift

Le script `simulate_production.py --drift` applique les décalages suivants :

| Feature | Décalage simulé | Impact attendu |
|---|---|---|
| `DAYS_BIRTH` | Clients plus âgés (+12 ans en moyenne) | Changement de profil démographique |
| `AMT_INCOME_TOTAL` | Revenus réduits de 20 à 40% | Profils financièrement plus fragiles |
| `AMT_CREDIT` | Durée de crédit allongée (36–96 mois vs 12–72) | Montants empruntés plus élevés |
| `AMT_ANNUITY` | Taux d'annuité plus élevé (5–8.5% vs 2.5–5.5%) | Charge de remboursement plus lourde |
| `EXT_SOURCE_1/2/3` | Scores externes décalés de -0.18 | Historique crédit plus risqué |
| `score` | Score de défaut plus élevé (attendu) | Conséquence directe du drift des features |

### Ce que ça signifie en production

Si ce drift se produisait réellement, l'API recevrait des profils **systématiquement plus risqués** que ceux sur lesquels le modèle a été entraîné. Deux risques :

1. **Sous-estimation du risque** : le modèle a été calibré sur des distributions différentes, ses probabilités de défaut pourraient être sous-estimées pour ces nouveaux profils.
2. **Biais de sélection** : le taux de rejet augmente mécaniquement, ce qui peut fausser les métriques métier.

### Actions recommandées

- **Court terme** : surveiller le taux de rejet et la distribution des scores hebdomadairement.
- **Moyen terme** : si le drift est confirmé sur données réelles, déclencher un ré-entraînement du modèle sur un jeu de données incluant les nouveaux profils.
- **Seuil d'alerte** : un drift détecté sur plus de 3 features clés (`EXT_SOURCE`, `AMT_INCOME_TOTAL`, `DAYS_BIRTH`) doit déclencher une revue manuelle.